# Расстояния и голосование

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Находить ближайших соседей и получать однозначный ответ по заранее заданным правилам.

## Идея метода

KNN (*k-nearest neighbors*, метод k ближайших соседей) ищет в train $k$ объектов, ближайших к новому. Класс, встречающийся у них чаще всего, становится ответом. При $k=1$ ответ совпадает с меткой одного ближайшего объекта.

KNN запоминает обучающие примеры; отдельного подбора коэффициентов здесь нет. Но выбор признаков, масштаба, расстояния и $k$ влияет на результат. В примере используем евклидово расстояние и равные голоса.

## Ручной расчёт

Новый учебный планер $q=(15,10)$. Для первых четырёх строк train квадраты расстояний:

| Объект | Вектор | Класс | Квадрат расстояния |
| --- | --- | --- | --- |
| g01 | (12, 8) | короткий | 13 |
| g02 | (22, 10) | дальний | 49 |
| g03 | (14, 28) | короткий | 325 |
| g04 | (24, 30) | дальний | 481 |

Если **для примера** train состоит только из этих четырёх строк, при $k=1$ ответ — `короткий`. При $k=3$ голоса: два за `короткий`, один за `дальний`, ответ тот же. В полном эксперименте используются все 12 строк train, а не эта иллюстративная подтаблица.

Число соседей k — **гиперпараметр**, то есть настройка метода, которую выбирают по validation. Это отличается от построения модели по train: проверочные ответы помогают выбрать настройку, но не становятся обучающими соседями. В Python слово «параметр» также обозначает аргумент функции; по контексту важно различать эти смыслы.

Чтобы результат не зависел от случайности, при одинаковых расстояниях берём меньший ID; при равенстве голосов — первую метку по алфавиту. Чётное $k$ чаще приводит к равенству, но и нечётное не решает вопрос автоматически, если классов больше двух.

Перед расчётом проверяем, что k — целое число Python, а запрос содержит хотя бы один признак. Вызов `type(k)` возвращает тип: строка `"3"`, дробное `1.5` и логическое `True` не подходят вместо целого числа соседей. Затем обычным циклом проверяем совпадение числа координат. Идентификаторы обучающих объектов должны быть уникальны, признаки — конечные числа.

## Разобранный пример: воспроизводимое голосование

Сохраняем для каждого кандидата тройку: квадрат расстояния, id, метку. Сортировка кортежей сравнивает сначала первый элемент, при равенстве второй и затем третий. Поэтому равные расстояния разрешаются по id. Срез `[:k]` выбирает первые k соседей. При равных голосах заранее выбираем первую метку по алфавиту.

In [ ]:
def predict(query, examples, k):
    if type(k) != int or not 1 <= k <= len(examples):
        raise ValueError("k должен быть целым от 1 до числа обучающих объектов")
    if len(query) == 0:
        raise ValueError("Нужен хотя бы один признак")
    for row in examples:
        if len(row["x"]) != len(query):
            raise ValueError("Число признаков должно совпадать")
    ranked = []
    for row in examples:
        squared_distance = 0
        for j in range(len(query)):
            squared_distance += (query[j] - row["x"][j]) ** 2
        ranked.append((squared_distance, row["id"], row["y"]))
    ranked.sort()
    votes = {}
    for distance, identifier, label in ranked[:k]:
        votes[label] = votes.get(label, 0) + 1
    winner = None
    for label in sorted(votes):
        if winner is None or votes[label] > votes[winner]:
            winner = label
    return winner

examples = [
    {"id": "a", "x": [12., 8.], "y": "короткий"},
    {"id": "b", "x": [22., 10.], "y": "дальний"},
    {"id": "c", "x": [14., 28.], "y": "короткий"},
]
assert predict([15., 10.], examples, 1) == "короткий"


## Правила применения

Порядок и число координат должны совпадать у запроса и всех обучающих объектов. Сам запрос не должен попадать в train как пример самого себя. Большое k сглаживает отдельные ответы, но может учитывать слишком далёкие объекты; его выбирают на отдельной части. Идентификаторы нужны только для разрешения равенств, а не как признаки расстояния.

## Разобранный пример: ничья голосов

Два ближайших объекта имеют одинаковое расстояние, id a и b, разные метки. При k=1 берём a по id. При k=2 голоса равны, поэтому берём первую метку по алфавиту. Это два разных правила: одно определяет состав соседей, другое итог голосования.

В коде `winner = None` означает, что победитель ещё не выбран. Первый класс в `sorted(votes)` занимает это место; следующий заменит его только при строго большем числе голосов. Так одна и та же проверка реализует и поиск максимума, и заранее заданное правило ничьей.

## Типичные ошибки

Не используйте запрос как собственного обучающего соседа. Проверяйте длины и порядок признаков, отдельно определяйте ничью расстояний и ничью голосов.

## Задачи

### 1. Базовая
Пять соседей упорядочены по возрастанию расстояния; их метки A,B,B,A,B. Что получится при k=3?

<details>
<summary>Подсказка</summary>

Берите только первые три.

</details>



### 2. Базовая

Можно ли k=0?

<details>
<summary>Подсказка</summary>

Ни одного голоса не будет.

</details>



### 3. Прикладная

Что получится для тех же соседей при k=5?

<details>
<summary>Подсказка</summary>

Посчитайте все голоса.

</details>



### 4. Прикладная
Для `examples` из разобранного примера найдите класс запроса `[21., 10.]` при k=1. Сначала сравните квадраты расстояний вручную, затем проверьте функцию.

<details>
<summary>Подсказка</summary>

Разности координат с объектом b равны −1 и 0.

</details>



### 5. Олимпиадная

Почему k=1 на обучающих объектах часто даёт идеальный ответ?

<details>
<summary>Подсказка</summary>

Объект ближе всего к себе.

</details>



### 6. Олимпиадная

Достаточно ли нечётного k, чтобы гарантировать отсутствие равенства голосов в многоклассовой задаче?

<details>
<summary>Подсказка</summary>

Проверьте, что изменится, если классов больше двух.

</details>



## Итоги

KNN сравнивает запрос только с обучающими объектами и собирает голоса k ближайших. Число и порядок признаков, масштаб и оба правила разрешения ничьей входят в описание метода.

## Самопроверка

Разделите ничью расстояний и ничью голосов. Почему нечётное k не гарантирует отсутствие ничьей при трёх классах?

Для перехода дальше достаточно базовых и прикладных задач; олимпиадные можно отложить.


## Решения задач

### Задача 1

Первые три метки A,B,B дают два голоса B и один A, поэтому ответ B.

### Задача 2

Нет. При k=0 нет соседей и голосов; функция требует 1≤k≤числа обучающих объектов.

### Задача 3

При k=5 учитываются все пять: B имеет три голоса, A — два. Ответ B.

### Задача 4

Квадраты расстояний до a, b и c равны соответственно 85, 1 и 373. Ближайший — b, поэтому ответ `дальний`.

In [ ]:
assert predict([21., 10.], examples, 1) == "дальний"


Это проверка вычисления на маленьком примере, а не итоговая оценка качества на независимой выборке.

### Задача 5

Это запоминание. При совпадающих признаках и разных метках возможны ошибки даже здесь.

### Задача 6

Нет. Нечётность помогает лишь для двух классов без иных условий. Правило равенств всё равно нужно.